# 1. Datenerhebung

## Ziel
Wir erheben die Stromtarife über die offizielle ElCom-/LINDAS-Schnittstelle.

## Abgrenzung
- Tarifjahr: 2026
- Haushaltsprofil: H4 mit 4’500 kWh Jahresverbrauch
- Produkt: Standardprodukt

Die abgerufenen Rohdaten speichern wir unverändert, damit unsere
Datenerhebung nachvollziehbar bleibt.

In [1]:
%pip install requests pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import requests
import pandas as pd

print("Die Umgebung für die Datenerhebung ist bereit.")

Die Umgebung für die Datenerhebung ist bereit.


## 1.1 ElCom-Stromtarife abrufen

Wir fragen die offiziellen Tarifdaten über den LINDAS-SPARQL-Endpunkt ab.

Filter:
- Tarifjahr 2026
- Haushaltsprofil H4
- Standardprodukt

Wir übernehmen die Gemeinde-ID, die Betreiber-ID und den Gesamttarif.
Eine Gemeinde kann mehrere Betreiber und damit mehrere Tarifzeilen haben.
Fehlende Gesamttarife behalten wir zunächst für die spätere Datenprüfung.

In [4]:
# Offizieller SPARQL-Endpunkt
endpoint = "https://ld.admin.ch/query"

# Abfrage für Stromtarife 2026, H4, Standardprodukt
query = """
PREFIX cube: <https://cube.link/>
PREFIX schema: <http://schema.org/>
PREFIX strom: <https://energy.ld.admin.ch/elcom/electricityprice/dimension/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?gemeinde ?betreiber ?gesamttarif

FROM <https://lindas.admin.ch/elcom/electricityprice>
FROM <https://lindas.admin.ch/territorial>
FROM <https://lindas.admin.ch/fso/register>

WHERE {
    <https://energy.ld.admin.ch/elcom/electricityprice>
        a cube:Cube ;
        cube:observationSet/cube:observation ?beobachtung .

    ?beobachtung
        strom:period "2026"^^xsd:gYear ;
        strom:category ?kategorie ;
        strom:product
            <https://energy.ld.admin.ch/elcom/electricityprice/product/standard> ;
        strom:municipality ?gemeinde ;
        strom:operator ?betreiber .

    ?kategorie schema:name "H4" .

    OPTIONAL {
        ?beobachtung strom:total ?gesamttarif .
    }
}

ORDER BY ?gemeinde ?betreiber
"""

In [6]:
response = requests.post(
    endpoint,
    data={"query": query},
    headers={"Accept": "application/sparql-results+json"},
    timeout=120
)

response.raise_for_status()

elcom_raw = response.json()

print("HTTP-Status:", response.status_code)
print("Anzahl Tarifzeilen:",
      len(elcom_raw["results"]["bindings"]))

HTTP-Status: 200
Anzahl Tarifzeilen: 2268


In [7]:
# Werte aus der verschachtelten API-Antwort herauslesen.
rows = [
    {
        field: entry["value"]
        for field, entry in record.items()
    }
    for record in elcom_raw["results"]["bindings"]
]

# In eine pandas-Tabelle umwandeln.
df_elcom = pd.DataFrame(
    rows,
    columns=["gemeinde", "betreiber", "gesamttarif"]
)

df_elcom.head()

,gemeinde,betreiber,gesamttarif
0,https://ld.admin.ch/municipality/1,https://energy.ld.admin.ch/elcom/electricitypr...,24.136
1,https://ld.admin.ch/municipality/10,https://energy.ld.admin.ch/elcom/electricitypr...,24.136
2,https://ld.admin.ch/municipality/100,https://energy.ld.admin.ch/elcom/electricitypr...,24.136
3,https://ld.admin.ch/municipality/1001,https://energy.ld.admin.ch/elcom/electricitypr...,25.414
4,https://ld.admin.ch/municipality/1002,https://energy.ld.admin.ch/elcom/electricitypr...,25.414


## 1.2 Rohdaten speichern

Wir speichern die unveränderte ElCom-API-Antwort als JSON-Datei.
Zusätzlich dokumentieren wir den Abrufzeitpunkt, die Schnittstelle
und die verwendete Abfrage.

Damit bleibt nachvollziehbar, welche Daten unsere Analyse verwendet.

In [8]:
from pathlib import Path

print("Aktueller Arbeitsordner:", Path.cwd())

Aktueller Arbeitsordner: /Users/leyla.erikci/Desktop/data-analytics-group-05/notebooks


In [9]:
import json
from datetime import datetime, timezone

arbeitsordner = Path.cwd()

projektordner = (
    arbeitsordner.parent
    if arbeitsordner.name == "notebooks"
    else arbeitsordner
)

rohdaten_ordner = projektordner / "data" / "raw"
rohdaten_ordner.mkdir(parents=True, exist_ok=True)

# Eigener Dateiname je Abruf, damit frühere Abrufe erhalten bleiben.
abrufzeit = datetime.now(timezone.utc)
zeitstempel = abrufzeit.strftime("%Y%m%dT%H%M%S%fZ")

dateipfad = rohdaten_ordner / f"elcom_2026_h4_{zeitstempel}.json"

# Unveränderte Antwort des Servers speichern.
dateipfad.write_bytes(response.content)

# Abfrage und Angaben zum Abruf dokumentieren.
metadaten = {
    "abrufzeit_utc": abrufzeit.isoformat(),
    "endpoint": endpoint,
    "tarifjahr": 2026,
    "profil": "H4",
    "produkt": "standard",
    "anzahl_tarifzeilen": len(elcom_raw["results"]["bindings"]),
    "rohdaten_datei": dateipfad.name,
    "sparql_abfrage": query
}

metadatenpfad = rohdaten_ordner / f"elcom_2026_h4_{zeitstempel}_metadaten.json"

metadatenpfad.write_text(
    json.dumps(metadaten, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Rohdaten gespeichert:", dateipfad)
print("Metadaten gespeichert:", metadatenpfad)

Rohdaten gespeichert: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/raw/elcom_2026_h4_20261117T195442966207Z.json
Metadaten gespeichert: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/raw/elcom_2026_h4_20261117T195442966207Z_metadaten.json


## 1.3 Erste Datenprüfung

Wir prüfen die Anzahl der Tarifzeilen, die eindeutigen Gemeinden
und Betreiber sowie fehlende Werte und doppelte Einträge.

Eine Gemeinde kann mehrere Betreiber haben. Mehrere Tarifzeilen
für dieselbe Gemeinde sind deshalb nicht automatisch Fehler.

Tarifzeilen: Wie viele Einträge hat die API geliefert?

Eindeutige Gemeinden und Betreiber: Wie viele unterschiedliche Kennungen kommen vor?

Fehlende Werte: Welche Angaben müssen wir später behandeln?

Identische Zeilen: Gibt es vollständig wiederholte Einträge?

Doppelte Gemeinde-Betreiber-Kombinationen: Gibt es für dieselbe Kombination mehrere Einträge, die wir genauer prüfen müssen?

In [11]:
print("Anzahl Tarifzeilen:", len(df_elcom))
print("Eindeutige Gemeinden:", df_elcom["gemeinde"].nunique())
print("Eindeutige Betreiber:", df_elcom["betreiber"].nunique())

print("\nFehlende Werte je Spalte:")
print(df_elcom.isna().sum())

print("\nVollständig identische Zeilen:")
print(df_elcom.duplicated().sum())

print("\nDoppelte Kombinationen aus Gemeinde und Betreiber:")
print(
    df_elcom.duplicated(
        subset=["gemeinde", "betreiber"]
    ).sum()
)

Anzahl Tarifzeilen: 2268
Eindeutige Gemeinden: 2122
Eindeutige Betreiber: 551

Fehlende Werte je Spalte:
gemeinde       0
betreiber      0
gesamttarif    1
dtype: int64

Vollständig identische Zeilen:
0

Doppelte Kombinationen aus Gemeinde und Betreiber:
0


### Ergebnis der ersten Datenprüfung

Der ElCom-Abruf enthält 2’268 Tarifzeilen für 2’122 eindeutige
Gemeinden und 551 Betreiber.

Gemeinde- und Betreiberkennungen sind vollständig vorhanden.
Bei einer Tarifzeile fehlt der Gesamttarif.

Es gibt weder vollständig identische Zeilen noch doppelte
Gemeinde-Betreiber-Kombinationen.

Mehrere Tarifzeilen derselben Gemeinde bleiben erhalten, da eine
Gemeinde mehrere Betreiber haben kann. Den fehlenden Gesamttarif
prüfen wir bei der Datenaufbereitung.

In [12]:
# Nur Zeilen anzeigen, bei denen der Gesamttarif fehlt.
fehlende_tarife = df_elcom[
    df_elcom["gesamttarif"].isna()
]

# Lange Kennungen vollständig anzeigen.
with pd.option_context("display.max_colwidth", None):
    display(fehlende_tarife)

,gemeinde,betreiber,gesamttarif
699,https://ld.admin.ch/municipality/3218,https://energy.ld.admin.ch/elcom/electricityprice/operator/326,NaN


## 1.4 BFS-Gemeindedaten erheben

Wir rufen die offiziellen BFS-Raumgliederungen zum Gemeindestand
01.01.2026 ab.

Wir benötigen die Gemeindenummer, den Gemeindenamen, den Kanton
und die Stadt/Land-Typologie 2020.

Die Gemeindenummer dient später als Schlüssel zur Verknüpfung
mit den ElCom-Tarifen.

In [13]:
bfs_url = (
    "https://www.agvchapp.bfs.admin.ch"
    "/api/communes/levels?date=01-01-2026"
)

bfs_response = requests.get(bfs_url, timeout=120)
bfs_response.raise_for_status()

# Abrufzeitpunkt dokumentieren.
bfs_abrufzeit = datetime.now(timezone.utc)
bfs_zeitstempel = bfs_abrufzeit.strftime("%Y%m%dT%H%M%S%fZ")

# Unveränderte CSV-Antwort speichern.
bfs_dateipfad = (
    rohdaten_ordner
    / f"bfs_gemeinden_2026_{bfs_zeitstempel}.csv"
)

bfs_dateipfad.write_bytes(bfs_response.content)

print("HTTP-Status:", bfs_response.status_code)
print("Rohdaten gespeichert:", bfs_dateipfad)

HTTP-Status: 200
Rohdaten gespeichert: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/raw/bfs_gemeinden_2026_20261117T195841888739Z.csv


In [14]:
df_bfs = pd.read_csv(bfs_dateipfad, encoding="utf-8-sig")

print("Anzahl Zeilen:", len(df_bfs))
print("\nVerfügbare Spalten:")
print(df_bfs.columns.tolist())

df_bfs.head()

Anzahl Zeilen: 2110

Verfügbare Spalten:
['HistoricalCode', 'BfsCode', 'Name', 'CantonId', 'Canton', 'DistrictId', 'District', 'FUA2014', 'DEGURB2021', 'FUA2021', 'GCITIES2021', 'STREG2022', 'DEGURB2011', 'GCITIES2011', 'REGCH', 'AGKG2020', 'STADTE2020', 'STGD2020', 'RSC2020', 'SPRGEB2020', 'AGKSA2020', 'AGGL2020', 'AGGLGK2020', 'MONT2019', 'STALAN2020', 'GDETYP2020_9', 'GDETYP2020_25', 'GBAE2018', 'BAE2018']


,HistoricalCode,BfsCode,Name,CantonId,Canton,DistrictId,District,FUA2014,DEGURB2021,FUA2021,...,SPRGEB2020,AGKSA2020,AGGL2020,AGGLGK2020,MONT2019,STALAN2020,GDETYP2020_9,GDETYP2020_25,GBAE2018,BAE2018
0,10078,6158,Vionnaz,23,Valais / Wallis,10013,District de Monthey,0,3,5586960,...,2,0,0,0,1,2,22,226,2,2040
1,10157,3023,Speicher,15,Appenzell Ausserrhoden,10098,Bezirk Mittelland,3203960,2,3203960,...,1,3203910,3203,3,1,2,21,217,15,15050
2,10159,6011,Zwischbergen,23,Valais / Wallis,10035,Bezirk Brig,0,3,0,...,1,0,0,0,1,3,33,338,7,7090
3,10162,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,10104,District de la Sarine,0,1,2196960,...,2,2196910,2196,4,0,1,12,122,4,4030
4,10165,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg,10104,District de la Sarine,0,3,2196960,...,2,2196910,2196,4,0,3,23,235,4,4030


In [15]:
bfs_metadaten = {
    "abrufzeit_utc": bfs_abrufzeit.isoformat(),
    "quelle": "Bundesamt für Statistik",
    "url": bfs_url,
    "gemeindestand": "2026-01-01",
    "typologie": "Stadt/Land-Typologie 2020",
    "anzahl_zeilen": len(df_bfs),
    "spalten": df_bfs.columns.tolist(),
    "rohdaten_datei": bfs_dateipfad.name
}

bfs_metadatenpfad = (
    rohdaten_ordner
    / f"bfs_gemeinden_2026_{bfs_zeitstempel}_metadaten.json"
)

bfs_metadatenpfad.write_text(
    json.dumps(bfs_metadaten, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("BFS-Metadaten gespeichert:", bfs_metadatenpfad)

BFS-Metadaten gespeichert: /Users/leyla.erikci/Desktop/data-analytics-group-05/data/raw/bfs_gemeinden_2026_20261117T195841888739Z_metadaten.json


In [16]:
df_bfs_auswahl = df_bfs[
    ["BfsCode", "Name", "CantonId", "Canton", "STALAN2020"]
].copy()

df_bfs_auswahl.head()

,BfsCode,Name,CantonId,Canton,STALAN2020
0,6158,Vionnaz,23,Valais / Wallis,2
1,3023,Speicher,15,Appenzell Ausserrhoden,2
2,6011,Zwischbergen,23,Valais / Wallis,3
3,2228,Villars-sur-Glâne,10,Fribourg / Freiburg,1
4,2230,Villarsel-sur-Marly,10,Fribourg / Freiburg,3


In [17]:
print("Anzahl BFS-Gemeinden:", len(df_bfs_auswahl))

print("\nFehlende Werte:")
print(df_bfs_auswahl.isna().sum())

print("\nDoppelte Gemeindenummern:")
print(df_bfs_auswahl["BfsCode"].duplicated().sum())

print("\nVerteilung der Stadt/Land-Codes:")
print(
    df_bfs_auswahl["STALAN2020"]
    .value_counts(dropna=False)
    .sort_index()
)

Anzahl BFS-Gemeinden: 2110

Fehlende Werte:
BfsCode       0
Name          0
CantonId      0
Canton        0
STALAN2020    0
dtype: int64

Doppelte Gemeindenummern:
0

Verteilung der Stadt/Land-Codes:
STALAN2020
1     509
2     580
3    1021
Name: count, dtype: int64
